# RabTech AI/ML Internship — Task 02
## Customer Churn: Baseline Notebook

**Purpose:** Explore the supplied starter dataset and compare a majority-class baseline with Logistic Regression.

**Important limitation:** The dataset has only 12 records. Results are educational and exploratory, not reliable estimates of real-world performance. The data source's consent, license, sampling method, and feature timing remain unverified.

## 1. Load the dataset

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/customer-churn-training.csv")
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
print("\\nColumns:", df.columns.tolist())

## 2. Data quality checks

In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))

print("Exact duplicate rows:", df.duplicated().sum())
print("\\nTarget distribution:")
display(df["churned"].value_counts().sort_index().to_frame("count"))

print("\\nPlan distribution:")
display(df["plan_type"].value_counts().to_frame("count"))

assert "churned" in df.columns
assert set(df["churned"].dropna().unique()).issubset({0, 1})

## 3. Define features and target

`churned` is the target. `customer_id` is excluded because it is an identifier, not a meaningful predictive feature. Feature meanings are inferred from column names and require provider confirmation.

In [ ]:
target = "churned"
excluded = ["customer_id", target]
X = df.drop(columns=excluded)
y = df[target].astype(int)

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Records:", len(df), "| Churned:", int(y.sum()), "| Not churned:", int((y == 0).sum()))

## 4. Majority-class baseline

The baseline predicts the most frequent target class. Leave-One-Out cross-validation (LOOCV) is used for an educational comparison, but 12 records are far too few for stable performance estimates. Each fold is highly sensitive to individual records.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import (
    confusion_matrix, classification_report, precision_score,
    recall_score, f1_score, balanced_accuracy_score
)

cv = LeaveOneOut()
baseline = DummyClassifier(strategy="most_frequent")
baseline_pred = cross_val_predict(baseline, X, y, cv=cv)

def show_metrics(name, y_true, y_pred):
    print(name)
    print("Confusion matrix (labels 0, 1):")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 3))
    print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 3))
    print("F1-score:", round(f1_score(y_true, y_pred, zero_division=0), 3))
    print("Balanced accuracy:", round(balanced_accuracy_score(y_true, y_pred), 3))
    print(classification_report(y_true, y_pred, labels=[0, 1], zero_division=0))

show_metrics("Majority-class baseline — LOOCV", y, baseline_pred)

## 5. Logistic Regression comparison

Preprocessing is placed inside a pipeline so it is fitted separately within each training fold. The comparison remains exploratory; a score on 12 records cannot establish generalization.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

model_pred = cross_val_predict(model, X, y, cv=cv)
show_metrics("Logistic Regression — LOOCV", y, model_pred)

## 6. Decision and limitations

- Compare Logistic Regression with the majority-class baseline; do not select a production model from these results.
- A false positive may lead to unnecessary retention contact; a false negative may miss a customer who needs help.
- Before real use, define the prediction date and horizon, verify that every feature is available at that time, and obtain a larger representative dataset.
- The data file does not document consent, license, provenance details, sampling, or label timing. Confirm these with the provider before redistribution or operational use.
- Human review is required for uncertain or invalid inputs. Keep ordinary customer support available regardless of model output.
- Monitor data quality, prediction rates, error rates, complaints, and unfair outcomes. Suspend the model and revert to human-led practices if safeguards fail.

**Conclusion:** This notebook demonstrates a baseline workflow only. It does not establish that churn can be predicted reliably in real customer populations.